In [4]:
import pandas as pd

from string import punctuation

import spacy

In [2]:
drinks = pd.read_json(r"Data/drinks.json")
drink_names = {drinks["id"][i]: drinks["aliases"][i] for i in range(len(drinks["id"]))}

In [3]:
class Sentence:

    def __init__(self, sentence, nlp):

        self.sentence = sentence
        self.nlp = nlp 



    def clean(self):

        sentence = self.sentence.lower()
        nlp = self.nlp

        #turn words into their main form
        doc = nlp(sentence)
        words = []

        for word in doc:
            if ' ' not in str(word): #check for extra spaces
                words.append(word.lemma_)
        
        ' '.join(words)

        translator = str.maketrans('', '', punctuation)
        cleaned_sentence = sentence.translate(translator)


        self.tokens = cleaned_sentence.split()

        return cleaned_sentence.split()



    def change_drink_to_id(self):

        tokens = self.tokens

        references = []
        reference_ids = []
        
        border = 0

        for i in range(len(tokens)):

            for drink_name, drink_val in drink_names.items():

                if i != 0 and i == border:
                    break

                #check if sny drink name is mentioned
                if tokens[i] in drink_val: 

                    #check if the drink is iced, to replace "iced" and the drink name with a single word
                    if i > 0 and tokens[i-1] == "iced":
                        tokens[i] = f"iced_{drink_name}"

                        references.append(tokens[i])
                        reference_ids.append(i-1)

                        tokens.pop(i-1)
                        tokens.append('')
                        border-=1 #border avoids empty indexses, since an elemet has been removed

                    else:
                        tokens[i] = drink_name
                        references.append(tokens[i])
                        reference_ids.append(i)
                    

        if border:
            tokens = tokens[0:border]


        self.tokens = tokens
        self.references = reference_ids

        return tokens, reference_ids

In [6]:
nlp = spacy.load("en_core_web_sm")

Sentence1 = Sentence("I would like a hot americano", nlp)

Sentence1.clean()
Sentence1.change_drink_to_id()

(['i', 'would', 'like', 'a', 'hot', 'americano'], [5])